# Projet de Détection d'Anomalies — Infrastructure IT

## 1. Contexte

Après nos projets de classification (churn) et de forecasting (ventes), nous abordons la **détection d'anomalies**.

**Objectif :** Détecter automatiquement les incidents sur une infrastructure IT avant qu'ils impactent les utilisateurs.

**Données :** `ANOMALY_DETECTION.PUBLIC.ANOMALY_RAW` — 83,334 observations (agrégées par minute), 41 colonnes.

**Principe :** On apprend le comportement **normal** du système, puis on signale toute minute qui s'écarte significativement de ce comportement. C'est comme un gardien qui connaît parfaitement la routine et détecte tout ce qui sort de l'ordinaire.

**Dataset CATS (Controlled Anomalies Time Series) :**
- 3 composants (A, B, C) qui communiquent en temps réel
- 13 capteurs × 3 agrégations (mean, max, std) = 39 features + timestamp + label
- 4% d'anomalies (incidents simulés avec cascade entre composants)
- Un incident typique : stimulus externe anormal → composant A déstabilisé → B surchargé → C dégradé

---
## 2. Méthodologie

### Approche 1 — Snowflake ML Anomaly Detection (Natif)
- Solution **100% SQL** avec `SNOWFLAKE.ML.ANOMALY_DETECTION`
- On entraîne sur les données **normales uniquement** (le modèle apprend la routine)
- Puis on détecte les anomalies sur des données non vues
- Le modèle utilise un **Gradient Boosting** avec auto-régression et génère des features temporelles (lags, tendance, heure)

### Approche 2 — Modèles Custom Python
- **Isolation Forest** (non supervisé) : isole les points rares sans utiliser les labels
- **One-Class SVM** (non supervisé) : apprend la frontière du comportement normal
- **XGBoost Classifier** (supervisé) : utilise les labels (0/1) pour comparaison

### Split des données
- **TRAINNN** (23,041 minutes) : données normales → le modèle apprend le comportement normal
- **TESTTT** (5,760 minutes) : données avec anomalies → on évalue la détection
- Le split est **chronologique** (train = passé, test = futur)

### Métriques d'évaluation
- **Recall** : % d'anomalies réelles détectées (le plus important en sécurité)
- **Precision** : % d'alertes qui sont de vraies anomalies
- **F1-score** : équilibre entre recall et precision

---
## 3. Exploration des Données

In [ ]:
%%sql -r train_sample
-- Aperçu des données d'entraînement (minutes normales)
SELECT * FROM ANOMALY_DETECTION.PUBLIC.TRAINNN LIMIT 10

In [ ]:
%%sql -r test_sample
-- Aperçu des données de test (avec anomalies)
SELECT * FROM ANOMALY_DETECTION.PUBLIC.TESTTT LIMIT 10

In [ ]:
%%sql -r test_distribution
-- Distribution des anomalies dans le jeu de test
SELECT
    Y AS label,
    CASE WHEN Y = 0 THEN 'Normal' ELSE 'Anomalie' END AS type,
    COUNT(*) AS nb_minutes,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM ANOMALY_DETECTION.PUBLIC.TESTTT), 2) AS pct
FROM ANOMALY_DETECTION.PUBLIC.TESTTT
GROUP BY Y
ORDER BY Y

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from snowflake.snowpark.context import get_active_session

# Charger les données de test pour visualisation
session = get_active_session()
test_df = session.sql('SELECT TIMESTAMP, CSO1_MEAN, ASIN1_MEAN, AMUD_MEAN, Y FROM ANOMALY_DETECTION.PUBLIC.TESTTT ORDER BY TIMESTAMP').to_pandas()
test_df['TIMESTAMP'] = pd.to_datetime(test_df['TIMESTAMP'])

# 3 capteurs clés avec anomalies en rouge
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
sensors = ['CSO1_MEAN', 'ASIN1_MEAN', 'AMUD_MEAN']
titles = ['Capteur CSO1 (output)', 'Stimulus ASIN1 (externe)', 'Commande AMUD']

for i, (sensor, title) in enumerate(zip(sensors, titles)):
    normal = test_df[test_df['Y'] == 0]
    anomaly = test_df[test_df['Y'] == 1]
    axes[i].plot(normal['TIMESTAMP'], normal[sensor], '.', color='#1f77b4', markersize=2, label='Normal')
    axes[i].plot(anomaly['TIMESTAMP'], anomaly[sensor], '.', color='red', markersize=4, label='Anomalie')
    axes[i].set_title(title, fontsize=13)
    axes[i].legend(loc='upper right')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Test : {len(test_df)} minutes | Anomalies : {len(anomaly)} ({len(anomaly)*100/len(test_df):.1f}%)')

---
## 4. Approche 1 : Snowflake ML Anomaly Detection (Natif)

On entraîne sur `TRAINNN` (données normales), puis on détecte sur `TESTTT` (données avec anomalies).

Le modèle utilise toutes les features (39 colonnes) + génère ses propres features temporelles (lags, tendance, heure).

In [ ]:
%%sql -r create_train_view
-- Préparer la vue d'entraînement (format TIMESTAMP_NTZ requis)
CREATE OR REPLACE VIEW ANOMALY_DETECTION.PUBLIC.TRAINNN_V1 AS SELECT
    * EXCLUDE TIMESTAMP,
    TO_TIMESTAMP_NTZ(TIMESTAMP) AS TIMESTAMP_V1
FROM ANOMALY_DETECTION.PUBLIC.TRAINNN

In [ ]:
%%sql -r create_test_view
-- Préparer la vue de test
CREATE OR REPLACE VIEW ANOMALY_DETECTION.PUBLIC.TESTTT_V1 AS SELECT
    * EXCLUDE TIMESTAMP,
    TO_TIMESTAMP_NTZ(TIMESTAMP) AS TIMESTAMP_V1
FROM ANOMALY_DETECTION.PUBLIC.TESTTT

In [ ]:
%%sql -r train_native
-- Entraîner le modèle natif
-- TARGET_COLNAME = 'Y' : on apprend à prédire le comportement normal de Y
-- LABEL_COLNAME = '' : approche non supervisée (pas de labels fournis)
-- Le modèle utilise toutes les autres colonnes comme features
CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION ANOMALY_DETECTION.PUBLIC.my_model(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'ANOMALY_DETECTION.PUBLIC.TRAINNN_V1'),
    TIMESTAMP_COLNAME => 'TIMESTAMP_V1',
    TARGET_COLNAME => 'Y',
    LABEL_COLNAME => ''
)

In [ ]:
%%sql -r detect_anomalies
-- Détecter les anomalies sur les données de test
BEGIN
    CALL ANOMALY_DETECTION.PUBLIC.my_model!DETECT_ANOMALIES(
        INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'ANOMALY_DETECTION.PUBLIC.TESTTT_V1'),
        TIMESTAMP_COLNAME => 'TIMESTAMP_V1',
        TARGET_COLNAME => 'Y',
        CONFIG_OBJECT => {'prediction_interval': 0.95}
    );
    LET x := SQLID;
    CREATE OR REPLACE TABLE ANOMALY_DETECTION.PUBLIC.My_anomalies AS SELECT * FROM TABLE(RESULT_SCAN(:x));
END

In [ ]:
%%sql -r confusion_matrix
-- Matrice de confusion : comparer prédictions vs labels réels
SELECT
    t.Y AS vrai_label,
    a.IS_ANOMALY AS predit_anomalie,
    COUNT(*) AS nb
FROM ANOMALY_DETECTION.PUBLIC.My_anomalies a
JOIN ANOMALY_DETECTION.PUBLIC.TESTTT t ON a.TS = t.TIMESTAMP
GROUP BY 1, 2
ORDER BY 1, 2

In [ ]:
import pandas as pd

# Calculer les métriques depuis la matrice de confusion
cm = confusion_matrix.copy()

# Extraire les valeurs
tn = int(cm[(cm['VRAI_LABEL'] == 0) & (cm['PREDIT_ANOMALIE'] == False)]['NB'].values[0])
fp = int(cm[(cm['VRAI_LABEL'] == 0) & (cm['PREDIT_ANOMALIE'] == True)]['NB'].values[0])
fn = int(cm[(cm['VRAI_LABEL'] == 1) & (cm['PREDIT_ANOMALIE'] == False)]['NB'].values[0])
tp = int(cm[(cm['VRAI_LABEL'] == 1) & (cm['PREDIT_ANOMALIE'] == True)]['NB'].values[0])

recall = tp / (tp + fn)
precision = tp / (tp + fp)
f1 = 2 * (precision * recall) / (precision + recall)

print('=== Métriques du Modèle Natif Snowflake ===')
print(f'  Recall    : {recall*100:.1f}% ({tp} anomalies détectées sur {tp+fn})')
print(f'  Precision : {precision*100:.1f}% ({tp} vraies alertes sur {tp+fp} alertes)')
print(f'  F1-score  : {f1:.4f}')
print(f'\n  Matrice de confusion :')
print(f'    Vrais Négatifs  (TN) : {tn}')
print(f'    Faux Positifs   (FP) : {fp}')
print(f'    Faux Négatifs   (FN) : {fn}')
print(f'    Vrais Positifs  (TP) : {tp}')

# Sauvegarder pour comparaison
native_metrics = {'Recall': recall, 'Precision': precision, 'F1': f1}

In [ ]:
%%sql -r native_features
-- Importance des features du modèle natif
CALL ANOMALY_DETECTION.PUBLIC.my_model!EXPLAIN_FEATURE_IMPORTANCE()

In [ ]:
# Afficher le top 10 des features les plus importantes
print('=== Top 10 Features (Modèle Natif) ===')
top_features = native_features.head(10)
for _, row in top_features.iterrows():
    print(f"  {row['RANK']:2.0f}. {row['FEATURE_NAME']:40s} score={row['SCORE']:.3f} ({row['FEATURE_TYPE']})")

---
## 5. Approche 2 : Modèles Custom Python

On entraîne 3 modèles :
- **Isolation Forest** : isole les points rares (non supervisé)
- **One-Class SVM** : apprend la frontière du normal (non supervisé)
- **XGBoost** : classification binaire (supervisé, utilise les labels)

Les modèles non supervisés sont entraînés uniquement sur les données normales.

In [ ]:
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session

# Charger les données
session = get_active_session()
train_df = session.sql('SELECT * FROM ANOMALY_DETECTION.PUBLIC.TRAINNN ORDER BY TIMESTAMP').to_pandas()
test_df = session.sql('SELECT * FROM ANOMALY_DETECTION.PUBLIC.TESTTT ORDER BY TIMESTAMP').to_pandas()

# Features = toutes les colonnes sauf TIMESTAMP et Y
feature_cols = [c for c in train_df.columns if c not in ['TIMESTAMP', 'Y']]

X_train = train_df[feature_cols].values
y_train = train_df['Y'].values
X_test = test_df[feature_cols].values
y_test = test_df['Y'].values

print(f'Train : {len(train_df)} minutes ({(y_train==1).sum()} anomalies)')
print(f'Test  : {len(test_df)} minutes ({(y_test==1).sum()} anomalies)')
print(f'Features : {len(feature_cols)}')

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np

# Standardiser (important pour One-Class SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Contamination estimée (~4% d'anomalies)
contamination = 0.04

# --- Isolation Forest (non supervisé) ---
# Entraîné sur les données normales, détecte les points rares
iso_forest = IsolationForest(contamination=contamination, random_state=42, n_estimators=200)
iso_forest.fit(X_train_scaled)
iso_pred = (iso_forest.predict(X_test_scaled) == -1).astype(int)

print('=== Isolation Forest (non supervisé) ===')
print(classification_report(y_test, iso_pred, target_names=['Normal', 'Anomalie']))

# --- One-Class SVM (non supervisé) ---
# On utilise un sous-échantillon (SVM est lent sur grandes données)
np.random.seed(42)
sample_size = min(10000, len(X_train_scaled))
sample_idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)

ocsvm = OneClassSVM(kernel='rbf', gamma='auto', nu=contamination)
ocsvm.fit(X_train_scaled[sample_idx])
ocsvm_pred = (ocsvm.predict(X_test_scaled) == -1).astype(int)

print('=== One-Class SVM (non supervisé) ===')
print(classification_report(y_test, ocsvm_pred, target_names=['Normal', 'Anomalie']))

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# --- XGBoost Classifier (supervisé) ---
# Utilise les labels pour apprendre — avantage injuste mais intéressant pour comparaison
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    scale_pos_weight=len(y_train[y_train==0]) / max(len(y_train[y_train==1]), 1),
    random_state=42, eval_metric='logloss'
)
xgb_model.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=False)
xgb_pred = xgb_model.predict(X_test_scaled)

print('=== XGBoost Classifier (supervisé) ===')
print(classification_report(y_test, xgb_pred, target_names=['Normal', 'Anomalie']))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

# Calculer les métriques pour chaque modèle
model_names = ['Isolation Forest', 'One-Class SVM', 'XGBoost']
preds_list = [iso_pred, ocsvm_pred, xgb_pred]

custom_results = {}
for name, pred in zip(model_names, preds_list):
    custom_results[name] = {
        'F1': f1_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred)
    }

# Graphique comparatif
comp_df = pd.DataFrame(custom_results).T
comp_df.plot(kind='bar', figsize=(10, 5), rot=0, color=['#2ca02c', '#1f77b4', '#d62728'])
plt.title('Comparaison des Modèles Custom — Détection d\'Anomalies', fontsize=14)
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('=== Classement (par F1-score) ===')
print(comp_df.sort_values('F1', ascending=False).round(4).to_string())

### 5.1 Enregistrement du meilleur modèle dans le Registry

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

# Identifier le meilleur modèle par F1-score
best_name = max(custom_results, key=lambda x: custom_results[x]['F1'])
best_metrics_custom = custom_results[best_name]

if best_name == 'Isolation Forest':
    best_obj = iso_forest
    deps = ['scikit-learn']
elif best_name == 'One-Class SVM':
    best_obj = ocsvm
    deps = ['scikit-learn']
else:
    best_obj = xgb_model
    deps = ['xgboost']

# Créer le schéma ML_MODELS
session.sql('CREATE SCHEMA IF NOT EXISTS ANOMALY_DETECTION.ML_MODELS').collect()

registry = Registry(session=session, database_name='ANOMALY_DETECTION', schema_name='ML_MODELS')

sample_input = pd.DataFrame(X_test_scaled[:10], columns=feature_cols)

mv = registry.log_model(
    model=best_obj,
    model_name='ANOMALY_DETECTOR_BEST',
    version_name='V1',
    sample_input_data=sample_input,
    conda_dependencies=deps,
    metrics={'F1': float(best_metrics_custom['F1']), 'Precision': float(best_metrics_custom['Precision']), 'Recall': float(best_metrics_custom['Recall'])},
    comment=f'Meilleur modele anomaly detection: {best_name}'
)

print(f'Modele enregistre : {best_name}')
print(f'Registry : ANOMALY_DETECTION.ML_MODELS.ANOMALY_DETECTOR_BEST (V1)')
print(f'F1={best_metrics_custom["F1"]:.4f} | Recall={best_metrics_custom["Recall"]:.4f} | Precision={best_metrics_custom["Precision"]:.4f}')

---
## 6. Résultats, Coûts & Comparaison

In [ ]:
import pandas as pd

print('=' * 70)
print('   6.1 COMPARAISON DES PERFORMANCES')
print('=' * 70)

rows = []
rows.append({'Modèle': 'Snowflake Natif', 'Type': 'Non supervisé', 'Recall': f"{native_metrics['Recall']*100:.1f}%", 'Precision': f"{native_metrics['Precision']*100:.1f}%", 'F1': f"{native_metrics['F1']:.4f}"})
for name, m in custom_results.items():
    sup = 'Supervisé' if name == 'XGBoost' else 'Non supervisé'
    rows.append({'Modèle': name, 'Type': sup, 'Recall': f"{m['Recall']*100:.1f}%", 'Precision': f"{m['Precision']*100:.1f}%", 'F1': f"{m['F1']:.4f}"})

print(pd.DataFrame(rows).to_string(index=False))

print()
print('=' * 70)
print('   6.2 COMPARAISON DES COUTS')
print('=' * 70)

print(pd.DataFrame({
    'Critère': ['Code', 'Setup', 'Crédits', 'Maintenance', 'Labels requis', 'Feature engineering'],
    'Snowflake Natif': ['SQL', 'Minutes', 'Faible', 'Très faible', 'Non', 'Automatique (lags, tendance)'],
    'Custom Python': ['Python', 'Heures', 'Moyen', 'Élevée', 'Non (IF/SVM) ou Oui (XGB)', 'Manuel']
}).to_string(index=False))

print()
print('=' * 70)
print('   6.3 VERDICT')
print('=' * 70)
print(f'Natif Snowflake    : Recall={native_metrics["Recall"]*100:.1f}%, F1={native_metrics["F1"]:.4f}')
print(f'Meilleur custom    : {best_name}, Recall={best_metrics_custom["Recall"]*100:.1f}%, F1={best_metrics_custom["F1"]:.4f}')
print()
print('En anomaly detection, le RECALL est prioritaire : mieux vaut une fausse alerte que rater un incident.')
print()
print('Snowflake Natif  → Idéal pour monitoring temps réel, setup en minutes')
print('Isolation Forest → Bon compromis non supervisé, pas besoin de labels')
print('XGBoost          → Meilleur si labels disponibles, mais pas toujours le cas en prod')

---
## 7. Optimisation & Pour Aller Plus Loin

### Pistes d'optimisation
- **Multi-séries** : détecter les anomalies sur chaque capteur séparément avec `SERIES_COLNAME`
- **Labeled training** : fournir les labels au modèle natif (`LABEL_COLNAME => 'Y'`) pour améliorer la détection
- **Prediction interval** : ajuster le seuil (0.95, 0.99, 0.999) pour contrôler le ratio recall/precision
- **Ensemble** : combiner les prédictions de plusieurs modèles (vote majoritaire)
- **Feature selection** : retirer les features avec score 0 pour simplifier le modèle

### Pour aller plus loin
- **Automatisation** : Snowflake Tasks pour ré-entraîner et détecter en continu
- **Alertes** : Snowflake Alerts pour envoyer un email/webhook quand une anomalie est détectée
- **Root cause analysis** : identifier quel capteur déclenche l'anomalie en premier
- **Dashboard** : Streamlit in Snowflake pour visualiser les anomalies en temps réel
- **Corrélation** : analyser la cascade entre composants (A → B → C) lors d'un incident